# Interactive 3D NIfTI Viewer

**경로만 바꾸고 모든 셀 Run All** — axial / coronal / sagittal 슬라이더가 뜹니다.

- Cyan 선 = 해당 뷰에서 다른 두 슬라이스의 교차선
- Win min/max 슬라이더로 window/level 조절 (HU 범위, float 모두 지원)


In [ ]:
# ─── 여기만 바꾸면 됩니다 ───────────────────────────────────────────────────
VOLUME_PATH = "/workspace/data/test_generateCT/smoke_33f.nii.gz"
# ───────────────────────────────────────────────────────────────────────────
#
# 예시 경로:
#   "/workspace/data/test_generateCT/generatect_lowres.nii.gz"   ← 128³ low-res
#   "/workspace/data/test_generateCT/generatect_hires.nii.gz"   ← 512² super-res
#   "/workspace/data/generatect_inference/runs/.../viral_pneumonia.nii.gz"
#   "/workspace/datasets/datasets/CT-RATE/dataset/valid_fixed/.../xxx.nii.gz"

In [ ]:
import nibabel as nib
import numpy as np

img = nib.load(VOLUME_PATH)
vol = img.get_fdata()

# 여분 배치/채널 차원 제거 (B,C,D,H,W) → (D,H,W)
while vol.ndim > 3:
    vol = vol.squeeze(axis=0)

spacing = img.header.get_zooms()[:3]
d, h, w = vol.shape

print(f"File     : {VOLUME_PATH.split('/')[-1]}")
print(f"Shape    : {vol.shape}  (axis-0 × axis-1 × axis-2)")
print(f"Dtype    : {vol.dtype}")
print(f"Spacing  : {tuple(round(float(s), 3) for s in spacing)} mm")
print(f"Range    : [{vol.min():.4f},  {vol.max():.4f}]")
print(f"p1 / p99 : [{np.percentile(vol, 1):.4f},  {np.percentile(vol, 99):.4f}]")

File     : generatect_lowres.nii.gz
Shape    : (201, 128, 128)  (axis-0 × axis-1 × axis-2)
Dtype    : float64
Spacing  : (1.0, 1.0, 1.0) mm
Range    : [-1.0797,  1.6411]
p1 / p99 : [-0.9858,  0.6413]


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

_p1  = float(np.percentile(vol, 1))
_p99 = float(np.percentile(vol, 99))
_vmin, _vmax = float(vol.min()), float(vol.max())
_step = (_vmax - _vmin) / 200

@interact(
    z    = IntSlider(min=0, max=d-1, value=d//2, step=1,      description='Axial  z',   continuous_update=True, layout={'width':'500px'}),
    y    = IntSlider(min=0, max=h-1, value=h//2, step=1,      description='Coronal y',  continuous_update=True, layout={'width':'500px'}),
    x    = IntSlider(min=0, max=w-1, value=w//2, step=1,      description='Sagittal x', continuous_update=True, layout={'width':'500px'}),
    wmin = FloatSlider(min=_vmin, max=_vmax, value=_p1,  step=_step, description='Win min', readout_format='.3f', layout={'width':'500px'}),
    wmax = FloatSlider(min=_vmin, max=_vmax, value=_p99, step=_step, description='Win max', readout_format='.3f', layout={'width':'500px'}),
)
def _view(z, y, x, wmin, wmax):
    if wmin >= wmax:
        wmax = wmin + _step

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Axial: vol[z, :, :]  — crosshairs at (y, x)
    axes[0].imshow(vol[z],        cmap='gray', vmin=wmin, vmax=wmax, aspect='equal', origin='upper')
    axes[0].axhline(y, color='cyan',   lw=0.7, alpha=0.7)
    axes[0].axvline(x, color='yellow', lw=0.7, alpha=0.7)
    axes[0].set_title(f'Axial   z = {z}')
    axes[0].set_xlabel('axis-2  →')
    axes[0].set_ylabel('axis-1  ↓')

    # Coronal: vol[:, y, :]  — crosshairs at (z, x)
    axes[1].imshow(vol[:, y, :],  cmap='gray', vmin=wmin, vmax=wmax, aspect='equal', origin='upper')
    axes[1].axhline(z, color='cyan',   lw=0.7, alpha=0.7)
    axes[1].axvline(x, color='yellow', lw=0.7, alpha=0.7)
    axes[1].set_title(f'Coronal  y = {y}')
    axes[1].set_xlabel('axis-2  →')
    axes[1].set_ylabel('axis-0  ↓')

    # Sagittal: vol[:, :, x]  — crosshairs at (z, y)
    axes[2].imshow(vol[:, :, x],  cmap='gray', vmin=wmin, vmax=wmax, aspect='equal', origin='upper')
    axes[2].axhline(z, color='cyan',   lw=0.7, alpha=0.7)
    axes[2].axvline(y, color='yellow', lw=0.7, alpha=0.7)
    axes[2].set_title(f'Sagittal x = {x}')
    axes[2].set_xlabel('axis-1  →')
    axes[2].set_ylabel('axis-0  ↓')

    fig.suptitle(VOLUME_PATH.split('/')[-1], fontsize=10, color='dimgray')
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=100, description='Axial  z', layout=Layout(width='500px'), max=200), Int…

In [ ]:
# (선택) 현재 슬라이스 위치를 PNG로 저장 — z/y/x를 수동으로 지정
SNAP_Z, SNAP_Y, SNAP_X = d // 2, h // 2, w // 2

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
kw = dict(cmap='gray', vmin=_p1, vmax=_p99, aspect='equal', origin='upper')
axes[0].imshow(vol[SNAP_Z],         **kw); axes[0].set_title(f'Axial   z={SNAP_Z}')
axes[1].imshow(vol[:, SNAP_Y, :],   **kw); axes[1].set_title(f'Coronal  y={SNAP_Y}')
axes[2].imshow(vol[:, :, SNAP_X],   **kw); axes[2].set_title(f'Sagittal x={SNAP_X}')
for ax in axes: ax.axis('off')
fig.suptitle(VOLUME_PATH.split('/')[-1], fontsize=10, color='dimgray')
plt.tight_layout()

snap_path = VOLUME_PATH.replace('.nii.gz', '_snapshot.png')
fig.savefig(snap_path, dpi=150, bbox_inches='tight')
print('snapshot saved:', snap_path)
plt.show()